In [5]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [6]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    # The first block can be a thinking block, so join only the text blocks.
    return "".join(block.text for block in message.content if block.type == "text")

In [7]:
from pydantic import BaseModel


class Task(BaseModel):
    task: str


# The schema root is an object, so the list of tasks is wrapped in one.
class Dataset(BaseModel):
    tasks: list[Task]


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Each item is a task that
requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 tasks.
"""

    response = client.messages.parse(
        model=model,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}],
        output_format=Dataset,
    )
    return [task.model_dump() for task in response.parsed_output.tasks]

In [8]:
import json

dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

dataset

[{'task': 'Write a Python function that takes an S3 bucket name and a prefix, and uses boto3 to return the total size in bytes of all objects under that prefix, handling pagination with a paginator.'},
 {'task': "Write a JSON IAM policy document that grants read-only access (s3:GetObject and s3:ListBucket) to a bucket named 'my-app-logs' and only to objects under the 'reports/' prefix, following least-privilege principles."},
 {'task': "Write a regular expression that matches a valid AWS EC2 instance ID, which is 'i-' followed by either 8 or 17 lowercase hexadecimal characters, and matches the entire string only."}]

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [10]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [11]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "This function sums object sizes under a prefix, using a paginator so it works for more than 1,000 objects:\n\n```python\nimport boto3\nfrom botocore.exceptions import BotoCoreError, ClientError\n\n\ndef get_prefix_size_bytes(bucket: str, prefix: str = \"\", s3_client=None) -> int:\n    \"\"\"\n    Return the total size in bytes of all objects under `prefix` in `bucket`.\n\n    Args:\n        bucket:    Name of the S3 bucket.\n        prefix:    Key prefix (e.g. \"logs/2024/\"). Empty string means the whole bucket.\n        s3_client: Optional preconfigured boto3 S3 client (useful for testing\n                   or custom credentials/regions).\n\n    Returns:\n        Total size of all matching objects, in bytes.\n\n    Raises:\n        ClientError / BotoCoreError on AWS or connection failures.\n    \"\"\"\n    s3 = s3_client or boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n\n    total_size = 0\n    try:\n        for page in paginator.